In [1]:
# journal_09_qualitative_print.ipynb -- one cell. Print-quality versions of the two qualitative figures.
# Same images, rows, columns, models (A, B s1, Thermal only), boxes (conf >= 0.25), colours, counts and titles
# as qualitative_*_v2.png. Only the figure size (174 mm wide), resolution (600 dpi), font (8 pt sans-serif)
# and box line width (0.8 pt) change. Titles are broken over 2-3 lines so they fit at 8 pt.
# Re-runs the three models on just these 8 images (a few seconds).
# OVERWRITES journal\figures\qualitative_flir_v2.png and qualitative_m3fd_v2.png, and writes
# qualitative_flir_v2.tif and qualitative_m3fd_v2.tif next to them.
import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

import lb_fusion as L

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ROWS = {"flir": ["FLIR_09403", "FLIR_08885", "FLIR_08934", "FLIR_09390"],
        "m3fd": ["01864", "01206", "00174", "03675"]}
A, B, T = "A (RGB only = detached adapter)", "B s1", "Thermal only"
CONF = 0.25
WIDTH_IN = 174 / 25.4              # 6.85 in
FONT_PT = 8
BOX_LW = 0.8                       # points
COLS = [("RGB + ground truth", None, "rgb"),
        ("A: RGB only", A, "rgb"),
        ("B: frozen RGB +\nthermal adapter", B, "rgb"),
        ("Thermal only\n(on thermal image)", T, "thermal")]

plt.rcParams.update({"font.family": "sans-serif",
                     "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
                     "font.size": FONT_PT, "axes.titlesize": FONT_PT, "legend.fontsize": FONT_PT})
colors = plt.cm.tab10(np.linspace(0, 1, 10))   # same colours as before

for name, rows in ROWS.items():
    ds = L.DATASETS[name]
    specs = L._analysis_specs(name)
    preds, gts = {}, None
    for m in (A, B, T):
        model = L._load_eval_model(specs[m], name, DEVICE)
        p, g, _, _ = L.collect_predictions(model, ds, rows, DEVICE, num_workers=0)
        preds[m], gts = p, (gts or g)
        del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    def hits(m, img_id):
        return L.per_image_hits(preds[m][img_id], gts[img_id], CONF)

    h0, w0 = cv2.imread(str(ds["root"] / "images" / "val" / f"{rows[0]}{ds['ext']}")).shape[:2]
    panel_h = (WIDTH_IN / 4) * h0 / w0
    title_h = 3 * FONT_PT * 1.45 / 72 + 0.06    # three title lines + padding
    fig, axes = plt.subplots(len(rows), 4, figsize=(WIDTH_IN, len(rows) * (panel_h + title_h) + 0.3),
                             layout="constrained")
    fig.get_layout_engine().set(w_pad=0.01, h_pad=0.03, wspace=0.01, hspace=0.01)

    for r, img_id in enumerate(rows):
        diff = hits(B, img_id) - hits(A, img_id)
        tag = f"fusion {'helps' if diff > 0 else 'hurts'}, {diff:+d}"
        rgb = cv2.cvtColor(cv2.imread(str(ds["root"] / "images" / "val" / f"{img_id}{ds['ext']}")), cv2.COLOR_BGR2RGB)
        th = cv2.imread(str(ds["thermal"]["val"] / f"{img_id}{ds['ext']}"), cv2.IMREAD_GRAYSCALE)
        th = cv2.resize(th, (rgb.shape[1], rgb.shape[0]))
        n_gt = len(gts[img_id][0])
        for c, (title, m, base) in enumerate(COLS):
            ax = axes[r, c]
            ax.imshow(rgb if base == "rgb" else th, cmap=None if base == "rgb" else "gray")
            if m is None:
                boxes, cls = gts[img_id]
                items = [(bx, int(k)) for bx, k in zip(boxes, cls)]
                text = f"{img_id}\n[{tag}]\n{title}"
            else:
                p = preds[m][img_id]
                p = p[p[:, 4] >= CONF] if len(p) else p
                items = [(d[:4], int(d[5])) for d in p]
                text = f"{title}\n({hits(m, img_id)}/{n_gt} found)"
            for bx, k in items:
                ax.add_patch(plt.Rectangle((bx[0], bx[1]), bx[2] - bx[0], bx[3] - bx[1], fill=False,
                                           edgecolor=colors[k % 10], linewidth=BOX_LW))
            ax.set_title(text, fontsize=FONT_PT, pad=2)
            ax.axis("off")
        print(f"{name} {img_id}: {tag}; found A {hits(A, img_id)}, B {hits(B, img_id)}, "
              f"thermal-only {hits(T, img_id)} of {n_gt}")

    handles = [plt.Line2D([0], [0], color=colors[k % 10], lw=1.5, label=v) for k, v in ds["names"].items()]
    fig.legend(handles=handles, loc="outside lower center", ncol=len(handles), frameon=False)

    out_png = L.JOURNAL / "figures" / f"qualitative_{name}_v2.png"
    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_png, dpi=600)
    try:
        fig.savefig(out_png.with_suffix(".tif"), dpi=600, pil_kwargs={"compression": "tiff_lzw"})
    except Exception as e:
        print(f"  (TIFF not saved: {e})")
    plt.close(fig)
    w, h = Image.open(out_png).size
    print(f"saved {out_png} ({w} x {h} px = {w / 600 * 25.4:.0f} x {h / 600 * 25.4:.0f} mm at 600 dpi)\n")

  batch 0/1...
  batch 0/1...
  batch 0/1...
flir FLIR_09403: fusion helps, +7; found A 2, B 9, thermal-only 10 of 11
flir FLIR_08885: fusion helps, +7; found A 3, B 10, thermal-only 11 of 15
flir FLIR_08934: fusion helps, +5; found A 7, B 12, thermal-only 14 of 14
flir FLIR_09390: fusion hurts, -1; found A 3, B 2, thermal-only 3 of 3
saved C:\Users\LENOVO\Documents\Thermal_Fusion_Paper\journal\figures\qualitative_flir_v2.png (4110 x 4772 px = 174 x 202 mm at 600 dpi)

  batch 0/1...
  batch 0/1...
  batch 0/1...
m3fd 01864: fusion helps, +3; found A 2, B 5, thermal-only 3 of 6
m3fd 01206: fusion helps, +3; found A 6, B 9, thermal-only 9 of 12
m3fd 00174: fusion helps, +2; found A 13, B 15, thermal-only 16 of 16
m3fd 03675: fusion hurts, -3; found A 15, B 12, thermal-only 8 of 16
saved C:\Users\LENOVO\Documents\Thermal_Fusion_Paper\journal\figures\qualitative_m3fd_v2.png (4110 x 4566 px = 174 x 193 mm at 600 dpi)

